# Typo detection from causal-language-model surprisal

A causal LM estimates (p(x_t\mid x_{<t})). Token surprisal, (-\log p(x_t\mid x_{<t})), is used as an anomaly signal. The experiment separates calibration from final evaluation because choosing a threshold on the test set would inflate accuracy.

In [2]:
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, precision_recall_curve, auc

from nlp_portfolio.generation import token_surprisals, calibrate_threshold

MODEL_ID = "gpt2"
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID).to(device).eval()
print("device:", device)

device: cpu


In [3]:
examples = [
    "The researcher published the results yesterday.",
    "The researhcer published the results yesterday.",
]
for sentence in examples:
    scores = token_surprisals(sentence, tokenizer, model)
    print(sentence)
    print([(token, round(value, 2)) for token, value in scores])

The researcher published the results yesterday.
[('Ġresearcher', 9.93), ('Ġpublished', 7.13), ('Ġthe', 1.53), ('Ġresults', 1.11), ('Ġyesterday', 6.64), ('.', 2.69)]
The researhcer published the results yesterday.
[('Ġresear', 12.52), ('h', 10.43), ('cer', 10.53), ('Ġpublished', 8.81), ('Ġthe', 1.78), ('Ġresults', 1.6), ('Ġyesterday', 6.88), ('.', 1.87)]


The logit at position (t-1) predicts token (t). Comparing a token to the logit at its own position leaks that token into the representation and yields an invalid score.

## Labeled data and sentence scores

Maximum surprisal is sensitive to one strong anomaly. The mean of the largest two token surprisals is more stable for multi-token misspellings.

In [5]:
DATA_URL = "https://raw.githubusercontent.com/elenipapadopulos/NLP_LAB3_Datasets/main/typo_dataset1.csv"
data = pd.read_csv(DATA_URL)

def sentence_score(text, top_n=2):
    values = sorted((value for _, value in token_surprisals(text, tokenizer, model)), reverse=True)
    return float(np.mean(values[:top_n])) if values else 0.0

data["score"] = [sentence_score(text) for text in data["text"]]
data[["text", "label", "score"]].head()

,text,label,score
0,"<s> I will meet you at the café tomorrow, and ...",1,6.763654
1,<s> She was feeling nervous about the presenta...,1,12.526418
2,<s> He didn’t like the idea of moving to a new...,1,12.667633
3,<s> They decided to take a walk in the park af...,1,8.135532
4,"<s> We need to finish the report by Friday, or...",1,11.425644


In [6]:
train_idx, test_idx = train_test_split(
    np.arange(len(data)), test_size=0.35, random_state=42, stratify=data["label"]
)
threshold = calibrate_threshold(data.loc[train_idx, "score"], data.loc[train_idx, "label"])
test_prediction = (data.loc[test_idx, "score"] >= threshold).astype(int)
print("calibration threshold:", round(threshold, 3))
print(classification_report(data.loc[test_idx, "label"], test_prediction, digits=3))

calibration threshold: 6.719
              precision    recall  f1-score   support

           0      0.000     0.000     0.000        10
           1      0.444     0.889     0.593         9

    accuracy                          0.421        19
   macro avg      0.222     0.444     0.296        19
weighted avg      0.211     0.421     0.281        19



In [7]:
precision, recall, _ = precision_recall_curve(data.loc[test_idx, "label"], data.loc[test_idx, "score"])
print("held-out PR-AUC:", round(auc(recall, precision), 3))

audit = data.loc[test_idx, ["text", "label", "score"]].copy()
audit["prediction"] = test_prediction
audit["correct"] = audit["label"] == audit["prediction"]
audit.sort_values(["correct", "score"]).head(12)

held-out PR-AUC: 0.297


,text,label,score,prediction,correct
15,"<s> I don't think she's going to make it, but ...",1,6.239396,0,False
51,<s> Weather may chanfe on any given day so it ...,0,11.114415,1,False
45,<s> I have been in touckh with most of these p...,0,11.143936,1,False
40,<s> The wind blue so hard that it nocked over ...,0,11.177486,1,False
46,<s> They told me that I had ben put on a deit ...,0,11.200150,1,False
44,<s> I should hasve called the police and foumn...,0,12.204808,1,False
32,<s> Yesterday I was thinking about the time I ...,0,12.571248,1,False
29,"<s> I don't now, the prolblem is that it can b...",0,12.857239,1,False
49,"<s> I met my bets freind on 2012-10-10, and we...",0,13.119638,1,False
33,"<s> He didhn't tell me what she was doing, so ...",0,14.078105,1,False


Rare names, scientific terminology and unusual syntax can be surprising without containing typos. Common real-word errors can remain probable in context. This detector is therefore an uncertainty signal, not a spelling oracle. A practical system should combine it with edit-distance candidates and a domain lexicon, then report recall at an operational false-positive rate.

Recorded CPU run: the held-out set contains only 19 sentences. Accuracy was 0.421 and PR-AUC was 0.297. The selected threshold recovered 8 of 9 positive examples but no negative examples. On this split, raw GPT-2 surprisal did not provide a useful typo classifier. The small test set and sensitivity to uncommon words limit the conclusion.